# Quantum Common Pool Resources

In [quantum_pd.ipynb](quantum_pd.ipynb) we saw that quantum entanglement acts as a **trusted mediator** in the two-player Prisoner's Dilemma: the EWL bracket J/J† enforces cooperation the way a regulator does, and the Q̂ strategy is the unique Nash equilibrium.

Here we ask: **does that analogy survive when we go to n players?**

We start from the fishery model in [common_pool.ipynb](common_pool.ipynb), discretize it into a three-player game, and quantize it with the natural n-player extension of EWL.

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt

# ── Pauli matrices ─────────────────────────────────────────────────────────────
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

# ── Fishery model parameters (from common_pool.ipynb) ─────────────────────────
y0, g0, g1, c0, s_init = 0.01, 0.03, 0.001, 0.125, 100.0

def y_t(s, b):   return y0 * s * np.sqrt(b)
def c_t(b):      return c0 * b

# ── Discrete strategy levels (3 symmetric players) ────────────────────────────
N = 3
b_C = (g0 / y0 * (1 - g1 * s_init))**2 / N   # sustainable boats per player
b_D = (y0 * s_init / c0)**2 / N               # competitive-eq boats per player
print(f"C-strategy: {b_C:.2f} boats/player")
print(f"D-strategy: {b_D:.2f} boats/player")

def profit(b_alice, b_others_total):
    """Alice's profit given her boats and total boats of the other two."""
    b_total = b_alice + b_others_total
    catch_total = y_t(s_init, b_total)
    alice_catch  = (b_alice / b_total) * catch_total
    return alice_catch - c_t(b_alice)

## Act 1 — The classical three-player fishery dilemma

We reduce the continuous fishery to a two-strategy game.  Each of the three fishers chooses:

| Strategy | Boats | Interpretation |
|----------|-------|----------------|
| **C** (cooperate) | $b_C = b_\text{sust}/3 \approx 2.4$ | Fish at the sustainable rate |
| **D** (defect) | $b_D = b_\text{comp}/3 \approx 21$ | Fish at the competitive equilibrium rate |

The catch function $y = y_0 s \sqrt{b_\text{total}}$ has strong diminishing returns: each additional boat reduces per-boat yield, so when all three fish intensively the individual profit hits zero.

> **Tragedy of the commons — CPR version:** The Nash equilibria are *asymmetric* — one player cooperates as the "sucker" while the other two overfish.  No equilibrium achieves the symmetric social optimum (C, C, C) where all earn 0.596.

This differs from the standard two-player PD where all-D is the unique NE.  In the fishery, the diminishing returns are so severe that switching from D to C is actually profitable when *both others* are already defecting — so (D, D, D) is *not* Nash.  But no one wants to be the lone cooperator in (C, D, D).

In [ ]:
# ── 8-outcome payoff table (Alice's profit in each (A, B, C) combination) ─────
strategies = {'C': b_C, 'D': b_D}
rows = []
for sA, sB, sC in itertools.product(['C', 'D'], repeat=3):
    bA, bB, bC_ = strategies[sA], strategies[sB], strategies[sC]
    pi_A = profit(bA, bB + bC_)
    pi_B = profit(bB, bA + bC_)
    pi_C = profit(bC_, bA + bB)
    rows.append((sA, sB, sC, pi_A, pi_B, pi_C))

print(f"{'Alice':>5} {'Bob':>5} {'Carol':>5}  {'π_A':>7} {'π_B':>7} {'π_C':>7}")
print("-" * 48)
for r in rows:
    print(f"{r[0]:>5} {r[1]:>5} {r[2]:>5}  {r[3]:>7.3f} {r[4]:>7.3f} {r[5]:>7.3f}")

# ── Nash equilibrium check ─────────────────────────────────────────────────────
def classical_payoff(sA, sB, sC):
    bA, bB, bC_ = strategies[sA], strategies[sB], strategies[sC]
    return profit(bA, bB + bC_), profit(bB, bA + bC_), profit(bC_, bA + bB)

nash = []
for sA, sB, sC in itertools.product(['C', 'D'], repeat=3):
    piA, piB, piC = classical_payoff(sA, sB, sC)
    alt_A = [classical_payoff(s, sB, sC)[0] for s in ['C','D']]
    alt_B = [classical_payoff(sA, s, sC)[1] for s in ['C','D']]
    alt_C = [classical_payoff(sA, sB, s)[2] for s in ['C','D']]
    if piA >= max(alt_A) and piB >= max(alt_B) and piC >= max(alt_C):
        nash.append((sA, sB, sC))

print(f"\nClassical Nash equilibria: {nash}")

## Act 2 — Quantizing the fishery: three-player EWL

The EWL protocol generalizes to $n$ players by giving each player one qubit and using an $n$-qubit entangling gate.  For three players:

$$J_3 = \frac{1}{\sqrt{2}}\bigl(I^{\otimes 3} + i\,\sigma_x \otimes \sigma_x \otimes \sigma_x\bigr)$$

This is the same formula as the two-player case, just with one more tensor factor.  It creates a **three-qubit GHZ-like entangled state** when applied to $|CCC\rangle$.

### Protocol

1. Start from $|CCC\rangle$ — all-cooperate computational basis state.
2. Apply $J_3$.
3. Each player applies their strategy unitary $U_A \otimes U_B \otimes U_C$.
4. Apply $J_3^\dagger$ to disentangle.
5. Measure; probabilities over the eight $|xyz\rangle$ basis states determine payoffs.

The three discrete strategies are the same as before:

| Strategy | Unitary | Action |
|----------|---------|--------|
| **C** | $I$ | Do nothing |
| **D** | $i\sigma_x$ | Bit-flip (classical defect) |
| **Q̂** | $i\sigma_z$ | Phase-flip (quantum cooperate) |

The payoff operator for Alice is diagonal in the $|CCC\rangle, |CCD\rangle, \ldots, |DDD\rangle$ basis, with entries given by her profit in each outcome.

In [ ]:
# ── 3-player EWL setup ─────────────────────────────────────────────────────────
I2 = np.eye(2, dtype=complex)
C_strat = I2.copy()
D_strat = 1j * sigma_x
Q_strat = 1j * sigma_z

J3 = (np.eye(8, dtype=complex) + 1j * np.kron(np.kron(sigma_x, sigma_x), sigma_x)) / np.sqrt(2)
J3_dag = J3.conj().T

CCC = np.array([1,0,0,0,0,0,0,0], dtype=complex)

# ── Payoff diagonals (basis order: CCC CCD CDC CDD DCC DCD DDC DDD) ────────────
# Alice's profit in each outcome: she contributes the first qubit
labels = ['CCC','CCD','CDC','CDD','DCC','DCD','DDC','DDD']

def alice_profit_for_label(lbl):
    sA, sB, sC_ = lbl[0], lbl[1], lbl[2]
    return profit(strategies[sA], strategies[sB] + strategies[sC_])

def bob_profit_for_label(lbl):
    sA, sB, sC_ = lbl[0], lbl[1], lbl[2]
    return profit(strategies[sB], strategies[sA] + strategies[sC_])

def carol_profit_for_label(lbl):
    sA, sB, sC_ = lbl[0], lbl[1], lbl[2]
    return profit(strategies[sC_], strategies[sA] + strategies[sB])

PA3 = np.array([alice_profit_for_label(l) for l in labels])
PB3 = np.array([bob_profit_for_label(l)  for l in labels])
PC3 = np.array([carol_profit_for_label(l) for l in labels])

def ewl3(UA, UB, UC):
    psi = J3_dag @ np.kron(np.kron(UA, UB), UC) @ J3 @ CCC
    probs = np.abs(psi)**2
    return probs @ PA3, probs @ PB3, probs @ PC3

# ── 27-entry payoff table ──────────────────────────────────────────────────────
strat_dict = {'C': C_strat, 'D': D_strat, 'Q': Q_strat}
strat_names = ['C', 'D', 'Q']

print(f"{'A':>3} {'B':>3} {'C':>3}  {'π_A':>7} {'π_B':>7} {'π_C':>7}  outcome")
print("-" * 56)
for sA, sB, sC in itertools.product(strat_names, repeat=3):
    pA, pB, pC = ewl3(strat_dict[sA], strat_dict[sB], strat_dict[sC])
    psi = J3_dag @ np.kron(np.kron(strat_dict[sA], strat_dict[sB]), strat_dict[sC]) @ J3 @ CCC
    dominant = labels[int(np.argmax(np.abs(psi)**2))]
    print(f"{sA:>3} {sB:>3} {sC:>3}  {pA:>7.3f} {pB:>7.3f} {pC:>7.3f}  |{dominant}⟩")

## A parity surprise: Q̂^⊗3 goes the wrong way

The 27-strategy table above shows something unexpected: there is **no pure Nash equilibrium**.  The (Q̂, Q̂, Q̂) entry gives **(0, 0, 0)** — the worst possible outcome, not the best.

The reason is pure algebra.  Recall from quantum_pd.ipynb that the two-player case works because:

$$\sigma_z^{\otimes 2} \;\text{commutes with}\; \sigma_x^{\otimes 2}$$

Two sign flips (one per anticommutation) cancel: $(-1)^2 = +1$ → Q̂^⊗2 passes through $J_2^\dagger(\cdot)J_2$ as if it were the identity, returning the state to $|CC\rangle$.

For three players the algebra is different:

$$\sigma_z^{\otimes 3} \;\textbf{anticommutes with}\; \sigma_x^{\otimes 3} \qquad [\text{three sign flips: }(-1)^3 = -1]$$

So $J_3^\dagger(\sigma_z^{\otimes 3})J_3 = +\sigma_y^{\otimes 3}$, which applied to $|CCC\rangle$ gives $-i|DDD\rangle$.  Three Q̂'s map **every** player to D — exactly the wrong direction.

### The parity rule

| Number of players $n$ | $\sigma_z^{\otimes n}$ vs $\sigma_x^{\otimes n}$ | $(Q̂,\ldots,Q̂)$ outcome |
|-----------------------|--------------------------------------------------|-------------------------|
| $n$ **even** (2, 4, 6, …) | commutes $(-1)^n = +1$ | $\rightarrow |CC\ldots C\rangle$ ✓ cooperative NE |
| $n$ **odd** (3, 5, 7, …) | anticommutes $(-1)^n = -1$ | $\rightarrow |DD\ldots D\rangle$ ✗ worst outcome |

The simplest fix: **use four fishers** ($n = 4$, even) where the parity works out.

In [ ]:
# ── Verify the parity rule ─────────────────────────────────────────────────────
# Odd n=3: σz⊗σz⊗σz anticommutes with σx⊗σx⊗σx
Sz3 = np.kron(np.kron(sigma_z, sigma_z), sigma_z)
Sx3 = np.kron(np.kron(sigma_x, sigma_x), sigma_x)
print("n=3: {σz^⊗3, σx^⊗3} = 0 (anticommute):", np.allclose(Sz3 @ Sx3 + Sx3 @ Sz3, 0))

psi_qqq_3 = J3_dag @ np.kron(np.kron(Q_strat, Q_strat), Q_strat) @ J3 @ CCC
print(f"n=3: (Q̂,Q̂,Q̂) outcome → |{labels[int(np.argmax(np.abs(psi_qqq_3)**2))]}⟩  (worst outcome!)")

# ── Four-player EWL: n=4 (even) — parity works ────────────────────────────────
N4 = 4
b_C4 = (g0 / y0 * (1 - g1 * s_init))**2 / N4
b_D4 = (y0 * s_init / c0)**2 / N4

def profit4(b_me, b_others):
    b_total = b_me + b_others
    return (b_me / b_total) * y_t(s_init, b_total) - c_t(b_me)

J4 = (np.eye(16, dtype=complex) + 1j * np.kron(np.kron(np.kron(sigma_x, sigma_x), sigma_x), sigma_x)) / np.sqrt(2)
J4_dag = J4.conj().T
CCCC = np.zeros(16, dtype=complex); CCCC[0] = 1.0

# Payoff labels for 4 players: CCCC, CCCD, ..., DDDD (16 states)
labels4 = [''.join(x) for x in itertools.product('CD', repeat=4)]
strats4  = {'C': b_C4, 'D': b_D4}

def alice_p4(lbl):
    return profit4(strats4[lbl[0]], sum(strats4[lbl[i]] for i in range(1,4)))

PA4 = np.array([alice_p4(l) for l in labels4])

def ewl4(UA, UB, UC, UD):
    psi = J4_dag @ np.kron(np.kron(np.kron(UA, UB), UC), UD) @ J4 @ CCCC
    return (np.abs(psi)**2 @ PA4)   # Alice's payoff (symmetric game)

# Verify n=4 parity
Sz4 = np.kron(np.kron(np.kron(sigma_z, sigma_z), sigma_z), sigma_z)
Sx4 = np.kron(np.kron(np.kron(sigma_x, sigma_x), sigma_x), sigma_x)
print("\nn=4: [σz^⊗4, σx^⊗4] = 0 (commute):", np.allclose(Sz4 @ Sx4 - Sx4 @ Sz4, 0))

psi_qqq4 = J4_dag @ np.kron(np.kron(np.kron(Q_strat, Q_strat), Q_strat), Q_strat) @ J4 @ CCCC
print(f"n=4: (Q̂,Q̂,Q̂,Q̂) outcome → |{labels4[int(np.argmax(np.abs(psi_qqq4)**2))]}⟩  (cooperative!)")

# Nash check for n=4 (by symmetry check Alice's best response; game is symmetric)
sd4 = {'C': C_strat, 'D': D_strat, 'Q': Q_strat}
def ewl4_full(sA, sB, sC, sD):
    psi = J4_dag @ np.kron(np.kron(np.kron(sd4[sA], sd4[sB]), sd4[sC]), sd4[sD]) @ J4 @ CCCC
    probs = np.abs(psi)**2
    return probs @ PA4

nash4 = []
for combo in itertools.product(['C','D','Q'], repeat=4):
    sA,sB,sC,sD = combo
    pA = ewl4_full(sA,sB,sC,sD)
    best_A = max(ewl4_full(s,sB,sC,sD) for s in ['C','D','Q'])
    # For full NE check use symmetry: all players face same game
    pB = ewl4_full(sB,sA,sC,sD)
    best_B = max(ewl4_full(s,sA,sC,sD) for s in ['C','D','Q'])
    pC = ewl4_full(sC,sA,sB,sD)
    best_C = max(ewl4_full(s,sA,sB,sD) for s in ['C','D','Q'])
    pD = ewl4_full(sD,sA,sB,sC)
    best_D = max(ewl4_full(s,sA,sB,sC) for s in ['C','D','Q'])
    if (np.isclose(pA,best_A) and np.isclose(pB,best_B) and
        np.isclose(pC,best_C) and np.isclose(pD,best_D)):
        nash4.append(combo)

print(f"\nn=4 quantum Nash equilibria: {nash4}")
R4 = ewl4_full('Q','Q','Q','Q')
print(f"Payoff at (Q̂,Q̂,Q̂,Q̂): {R4:.4f}  [compare R = {profit4(b_C4, 3*b_C4):.4f}]")

## What does this tell us about quantum regulation?

### The parity constraint is physical, not a bug

The failure of $n = 3$ is not a model artifact — it is a structural property of the EWL quantization with this gate.  The gate $J_n = \tfrac{1}{\sqrt{2}}(I^{\otimes n} + i\sigma_x^{\otimes n})$ produces cooperative NE only when $n$ is even.  For odd $n$, a different entangling gate is needed (e.g., one that creates GHZ states of a different parity, or a gate designed for the specific payoff structure).

This has no analogue in classical institution design: Ostrom's covenants work for 3 fishers just as well as 4.

### What still holds for even n

For $n = 4$ (and any even $n$), the structure from quantum_pd.ipynb extends perfectly:

| Property | 2 players | 4 players |
|----------|-----------|-----------|
| Gate | $J_2 = \tfrac{1}{\sqrt{2}}(I^{\otimes 2}+i\sigma_x^{\otimes 2})$ | $J_4 = \tfrac{1}{\sqrt{2}}(I^{\otimes 4}+i\sigma_x^{\otimes 4})$ |
| D passes through J | ✓ | ✓ |
| $(Q̂,\ldots,Q̂)$ → $|CC\ldots C\rangle$ | ✓ | ✓ |
| Unique symmetric NE | $(Q̂, Q̂)$ | $(Q̂, Q̂, Q̂, Q̂)$ |

### Entanglement encodes but does not eliminate the regulator

Even when EWL works ($n$ even), $J_n$ is still a **global gate** requiring all $n$ players to submit to the same apparatus.  The institution is not removed — it is encoded in physics.  The fishers must agree to enter the J, just as they must agree to enter Ostrom's quota system.

What quantum mechanics adds is that once the protocol is in place, defection is physically self-defeating (Q̂ converts D's gain into a collective loss), not merely socially penalized.

### The classical price of the quantum apparatus

The Toner-Bacon result scales with $n$: classically simulating the $n$-player maximally entangled state requires shared randomness plus $O(n^2)$ bits of communication.  The quantum apparatus becomes *relatively cheaper* as the fishery grows.

> **Ostrom's lesson survives quantum, with a twist:** you still need to build and trust the J, but the J must be designed for the right parity — and once built, it makes cooperation self-enforcing in a way no classical institution can match.

## Literature

Quantum CPR and public-goods games have been studied independently of this notebook:

- **Grau-Climent et al. (2023)** "Dynamics of a Quantum Common-Pool Resource Game with Homogeneous Players' Expectations." *Entropy* 25(12):1585. — First dedicated quantum CPR analysis; uses continuous-variable EWL quantization and studies stability and bifurcation of the Nash equilibrium under bounded rationality.

- **Grau-Climent et al. (2024)** "A quantum Stackelberg common-pool resource game." *APL Quantum* 1(3):036111. — Extends to asymmetric (leader-follower) timing, showing that quantum entanglement shifts the Stackelberg equilibrium toward cooperation.

- **Flitney & Abbott (2002)** "Quantum games with decoherence." Multi-player extension of EWL; shows the $n$-qubit gate generalizes naturally and Q̂-like strategies retain their Nash property under weak decoherence.

- **Li, Du & Massar (2002)** "Continuous-variable quantum games." Alternative quantization scheme for games with continuous strategy spaces (relevant to the continuous boat-choice in common_pool.ipynb).

- **Frąckiewicz (2011)** — already cited in quantum_pd.ipynb. The equivalence between discrete EWL and classical mediated games has not yet been fully extended to $n > 2$ players; this remains an open theoretical question.

### Connection to this notebook

The chain of equivalences from quantum_pd.ipynb extends one step further:

$$\text{EWL}_3 \xrightarrow{\text{van Enk-Pike}} \text{classical 4×4×4 mediated game} \xrightarrow{\text{Ostrom}} \text{institution-formation game}$$

The quantum apparatus ($J_3$) is the institution. Building it costs either a quantum device or, classically, shared randomness plus $O(n^2)$ bits — the price of cooperation without physics.